# 02 · EDA: interactions between features (Jadon, wide-and-deep)

**Why this notebook exists.** My model is wide-and-deep. Its *wide* part is a linear model on
**crossed features**: one weight for each combination, such as "Gold customer AND last order
31-60 days ago". A cross is only worth adding if the combination tells us something that the two
columns do not tell us on their own. That is called an **interaction**.

**The question.** Does the effect of one feature depend on the value of another? For example:
*does recency matter more for Gold than for Bronze customers?* If it does, a `customer_tier x
recency` cross lets the wide part learn a separate rule for each tier.

**What this notebook decides.** Which crosses go into the wide part. They are saved to
`results/wide_deep_crosses_jadon.json`, and `04_wide_deep_jadon.ipynb` reads them from there.

| Section | What it does |
|---|---|
| 0 | Setup, data, shared split |
| 1 | The headline question: recency x tier |
| 2 | A screen of every candidate pair, with a statistical test |
| 3 | A closer look at the strongest crosses, cell by cell |
| 4 | Decisions for the model |

**Only the train split is explored.** If I chose crosses by looking at validation or test
customers, the model's scores on them would be flattered. Validation is used once, in Section 2,
the same way it is used for tuning: to check that a pattern still holds on unseen customers.

## 0. Setup

In [ ]:
import json
import sys
from pathlib import Path

# Find the project root (the folder with src/data_prep.py), from the root or from notebooks/.
ROOT = Path.cwd()
while not (ROOT / "src" / "data_prep.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Shared team code: same data, same split as everyone else
from data_prep import CATEGORICAL_COLS, ID_COL, RESULTS_DIR, SEED, TARGET_COL, get_splits, load_data, split_xy
# My module: the bucket rules and the interaction test
from wide_deep import CROSS_SEP, RECENCY_COL, RECENCY_LABELS, interaction_test, recency_bucket

FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 50)
np.random.seed(SEED)

In [ ]:
df = load_data()
train, val, test = get_splits(df)        # the one shared 70/15/15 split, seed 42
X_train, y_train = split_xy(train)
X_val, y_val = split_xy(val)

base_rate = y_train.mean()
print(f"Train: {len(train):,} customers, {base_rate:.1%} buy  |  Validation: {len(val):,}")
print("Test split is not used in this notebook.")

## 1. Does recency matter more for Gold than for Bronze customers?

`days_since_last_order` is put into business buckets (0-30, 31-60, 61-90, 91-180, 181-365, 366+
and "never", which is coded as 1100 days). For each tier x bucket cell: the share of customers who
bought in October, and how many customers the cell has. A rate from a small cell is unreliable,
so the counts are shown next to it.

In [ ]:
eda = pd.DataFrame({
    "tier": X_train["customer_tier"],
    "recency": pd.Categorical(recency_bucket(X_train[RECENCY_COL]), categories=RECENCY_LABELS),
    "bought": y_train,
})
TIER_ORDER = [t for t in ["Bronze", "Silver", "Gold", "Platinum"] if t in set(eda["tier"])]
TIER_ORDER += sorted(set(eda["tier"]) - set(TIER_ORDER))   # any other tier name, just in case

rate = eda.pivot_table(index="tier", columns="recency", values="bought", aggfunc="mean", observed=False)
count = eda.pivot_table(index="tier", columns="recency", values="bought", aggfunc="size", observed=False)
rate, count = rate.loc[TIER_ORDER], count.loc[TIER_ORDER]

print("Buy rate by tier (rows) and recency bucket (columns), train split")
display(rate.style.format("{:.1%}", na_rep="-").background_gradient(cmap="Blues", axis=None))
print("Customers per cell")
display(count)

In [ ]:
# One line per tier: buy rate across the recency buckets, with a 95% error bar.
# The error bar is 1.96 x the standard error of a proportion, sqrt(p(1-p)/n).
colours = ["#a0704a", "#8e9aa6", "#d4a017", "#5b5b8f", "#444444", "#999999"]
fig, ax = plt.subplots(figsize=(9, 5))
for tier, colour in zip(TIER_ORDER, colours):
    p, n = rate.loc[tier], count.loc[tier]
    err = 1.96 * np.sqrt(p * (1 - p) / n.clip(lower=1))
    ax.errorbar(range(len(RECENCY_LABELS)), p, yerr=err, marker="o", capsize=3,
                linewidth=2, color=colour, label=tier)
ax.axhline(base_rate, color="#52514e", linestyle="--", linewidth=1, label=f"all customers ({base_rate:.1%})")
ax.set_xticks(range(len(RECENCY_LABELS)), RECENCY_LABELS)
ax.set_xlabel("Days since last order (bucket)")
ax.set_ylabel("Share who bought in October")
ax.set_title("Does recency matter more for some tiers?", loc="left", fontweight="bold")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e4e3df")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIGURES / "jadon_10_tier_x_recency.png", dpi=150)
plt.show()

**How to read the chart.** If recency mattered *equally* for every tier, the lines would be
parallel (each tier shifted up or down by the same amount). A line that **drops more steeply**
means recency matters more for that tier. Lines that are not parallel are an interaction, and a
cross can capture it.

The cell below puts a number on "steeper": for each tier, the buy rate of recent customers
(0-30 days) divided by the rate of lapsed customers (181-365 days).

In [ ]:
summary = pd.DataFrame({
    "recent 0-30": rate["0-30"],
    "lapsed 181-365": rate["181-365"],
    "customers": count.sum(axis=1),
})
summary["drop (points)"] = (summary["recent 0-30"] - summary["lapsed 181-365"]) * 100
summary["ratio recent / lapsed"] = summary["recent 0-30"] / summary["lapsed 181-365"]
display(summary.style.format({"recent 0-30": "{:.1%}", "lapsed 181-365": "{:.1%}",
                              "customers": "{:,}", "drop (points)": "{:.1f}",
                              "ratio recent / lapsed": "{:.1f}x"}))

if {"Gold", "Bronze"} <= set(summary.index):
    g, b = summary.loc["Gold"], summary.loc["Bronze"]
    print(f"Gold:   {g['recent 0-30']:.1%} of recent customers buy vs {g['lapsed 181-365']:.1%} of lapsed "
          f"-> {g['ratio recent / lapsed']:.1f}x")
    print(f"Bronze: {b['recent 0-30']:.1%} of recent customers buy vs {b['lapsed 181-365']:.1%} of lapsed "
          f"-> {b['ratio recent / lapsed']:.1f}x")
    more = "Gold" if g["drop (points)"] > b["drop (points)"] else "Bronze"
    print(f"Recency moves the buy rate by more percentage points for {more} customers.")

**My finding (write this in your own words after running):** which tier's line is steepest, how
big the gap is in percentage points, and whether the error bars overlap. Section 2 tests whether
the difference is larger than chance.

## 2. A screen of every candidate pair

Section 1 looked at one pair because the business asked about it. Many other pairs could interact
too, so they are all tested in the same way.

**Candidates.**
- Text columns with few levels: country, tier, region tier, channel, language. `region` is left
  out: 100 regions x 5 buckets = 500 cells, most of them too small to learn a rule from
  (Leon's embedding model handles region instead).
- Numeric columns: `days_since_last_order` plus the 5 numbers most related to buying on the train
  split (by Spearman correlation). Each is cut into 5 equal-sized buckets (quantiles).

**The test, for each pair (a, b)** (`interaction_test` in `src/wide_deep.py`). Two logistic
regressions on the bucketed columns:

| Model | Formula | Meaning |
|---|---|---|
| main effects | buy ~ a + b | each column adds its own amount |
| with cross | buy ~ a + b + a x b | every combination can have its own rate |

- **Deviance gain** and **p-value** (likelihood-ratio test on train). With 44,800 customers almost
  any pattern is "significant", so the p-value alone is too easy to pass.
- **Validation log-loss gain.** The practical test: does the cross still make predictions better
  for customers it was not fitted on? Above 0 = it helps. This is the number used to rank pairs.

In [ ]:
low_card_text = [c for c in CATEGORICAL_COLS if X_train[c].nunique() <= 10]
numeric = X_train.select_dtypes("number")
numeric = numeric.loc[:, numeric.nunique() > 1]

# Spearman correlation with the target, on TRAIN only. Ranks, so outliers do not dominate.
spearman = numeric.apply(lambda s: pd.Series(s).corr(pd.Series(y_train, index=s.index), method="spearman"))
top_numeric = spearman.abs().drop(RECENCY_COL, errors="ignore").sort_values(ascending=False).head(5)
num_candidates = [RECENCY_COL] + list(top_numeric.index)

print("Text columns:", low_card_text)
print("Numeric columns (|Spearman| with buying):")
for c in num_candidates:
    print(f"  {c:<30} {spearman[c]:+.3f}")

In [ ]:
from itertools import combinations

pairs = [(a, b) for a in low_card_text for b in num_candidates]   # text x number
pairs += list(combinations(low_card_text, 2))                       # text x text
pairs += list(combinations(num_candidates[:4], 2))                  # number x number (top 4)
print(f"Testing {len(pairs)} pairs...")

screen = pd.DataFrame([interaction_test(X_train, y_train, X_val, y_val, a, b) for a, b in pairs])
screen = screen.sort_values("val_logloss_gain", ascending=False).reset_index(drop=True)
screen["val_logloss_gain_x1000"] = screen["val_logloss_gain"] * 1000   # easier to read
display(screen[["cross", "extra_params", "deviance_gain", "p_value", "val_logloss_gain_x1000"]]
        .head(15).style.format({"deviance_gain": "{:.1f}", "p_value": "{:.2g}",
                                "val_logloss_gain_x1000": "{:+.2f}"}))

In [ ]:
top = screen.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 6))
colours_bar = np.where(top["val_logloss_gain"] > 0, "#2a78d6", "#c9c8c3")
ax.barh(top["cross"], top["val_logloss_gain_x1000"], color=colours_bar, height=0.7)
ax.axvline(0, color="#52514e", linewidth=1)
ax.set_xlabel("Validation log-loss improvement from the cross (x 1000)")
ax.set_title("Which combinations add information? (top 15 pairs)", loc="left", fontweight="bold")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
fig.tight_layout()
fig.savefig(FIGURES / "jadon_11_interaction_screen.png", dpi=150)
plt.show()

## 3. A closer look at the strongest crosses

A cross can win the screen because of one tiny cell with an extreme rate. Before trusting it,
look at every cell of the top three: the buy rate, and the number of customers behind it.

In [ ]:
from wide_deep import Bucketizer

def cross_table(col_a, col_b):
    # Buy rate and customer count for every combination of the two (bucketed) columns.
    b = Bucketizer().fit(X_train, [col_a, col_b]).transform(X_train)
    t = pd.DataFrame({"a": b[col_a], "b": b[col_b], "bought": y_train})
    rates = t.pivot_table(index="a", columns="b", values="bought", aggfunc="mean")
    sizes = t.pivot_table(index="a", columns="b", values="bought", aggfunc="size")
    if col_b == RECENCY_COL:
        order = [c for c in RECENCY_LABELS if c in rates.columns]
        rates, sizes = rates[order], sizes[order]
    if col_a == "customer_tier":                     # Bronze -> Platinum, not alphabetical
        order = [t for t in TIER_ORDER if t in rates.index]
        rates, sizes = rates.loc[order], sizes.loc[order]
    if col_b == "customer_tier":
        order = [t for t in TIER_ORDER if t in rates.columns]
        rates, sizes = rates[order], sizes[order]
    return rates, sizes

top3 = screen.head(3)
fig, axes = plt.subplots(1, len(top3), figsize=(6 * len(top3), 4.5))
for ax, (_, row) in zip(np.atleast_1d(axes), top3.iterrows()):
    rates, sizes = cross_table(row["col_a"], row["col_b"])
    im = ax.imshow(rates.to_numpy(dtype=float), cmap="Blues", vmin=0, aspect="auto")
    for i in range(rates.shape[0]):
        for j in range(rates.shape[1]):
            r, n = rates.iat[i, j], sizes.iat[i, j]
            if not np.isnan(r):
                ax.text(j, i, f"{r:.0%}\nn={n:.0f}", ha="center", va="center", fontsize=7,
                        color="white" if r > rates.max().max() * 0.6 else "black")
    ax.set_xticks(range(rates.shape[1]), rates.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(rates.shape[0]), rates.index, fontsize=8)
    ax.set_xlabel(row["col_b"])
    ax.set_ylabel(row["col_a"])
    ax.set_title(row["cross"], fontsize=10, loc="left")
fig.suptitle("Buy rate per cell (train split), with customers per cell", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
fig.savefig(FIGURES / "jadon_12_top_crosses.png", dpi=150)
plt.show()

**Check before moving on:** in each heatmap, is the pattern carried by cells with hundreds of
customers, or by a cell with fewer than about 30? The wide part merges any combination with fewer
than 30 train customers into one "infrequent" input, so tiny cells cannot become rules on their own.

## 4. Decisions for the model

**Rule.** Keep a cross if (1) it improves validation log loss (gain above 0) **and** (2) its
likelihood-ratio p-value is below 0.001. Keep at most 8, best first: each cross adds tens of
weights, and the wide part should stay small enough to read. `customer_tier x
days_since_last_order` is the headline question, so it is reported whether or not it is kept.

In [ ]:
MAX_CROSSES = 8
passed = screen[(screen["val_logloss_gain"] > 0) & (screen["p_value"] < 1e-3)]
chosen = passed.head(MAX_CROSSES)

headline = f"customer_tier{CROSS_SEP}{RECENCY_COL}"
in_screen = screen[screen["cross"] == headline]
if len(in_screen):
    r = in_screen.iloc[0]
    kept = "KEPT" if headline in set(chosen["cross"]) else "not kept"
    print(f"Headline cross {headline}: val gain x1000 = {r['val_logloss_gain_x1000']:+.2f}, "
          f"p = {r['p_value']:.2g} -> {kept}")

print(f"\n{len(passed)} of {len(screen)} pairs pass the rule; keeping {len(chosen)}:")
display(chosen[["cross", "extra_params", "p_value", "val_logloss_gain_x1000"]]
        .style.format({"p_value": "{:.2g}", "val_logloss_gain_x1000": "{:+.2f}"}))

decision = {
    "crosses": [[a, b] for a, b in zip(chosen["col_a"], chosen["col_b"])],
    "rule": "val_logloss_gain > 0 and p_value < 0.001, at most 8, ranked by val_logloss_gain",
    "screened_pairs": len(screen),
    "made_by": "notebooks/02_eda_interactions_jadon.ipynb",
}
out = RESULTS_DIR / "wide_deep_crosses_jadon.json"
out.write_text(json.dumps(decision, indent=2))
screen.to_csv(RESULTS_DIR / "wide_deep_interaction_screen_jadon.csv", index=False)
print("Saved:", out.relative_to(ROOT).as_posix())

### Findings (write these in your own words after running; you will be asked about them live)

1. **Recency x tier.** Does recency matter more for Gold than for Bronze? Quote the two ratios
   from Section 1 and say whether the screen kept the cross.
2. **The strongest crosses.** Name the top two or three and say, in business terms, what the rule
   is (for example, "WhatsApp works much better in SG than elsewhere").
3. **What was rejected and why.** Pairs that were "significant" on train but did not help on
   validation are the reason the validation check exists.
4. **What this decides.** The crosses above go into the wide part of `04_wide_deep_jadon.ipynb`.
   If the screen kept none, that is also a finding: the deep part alone may already capture the
   interactions, and the ablation in notebook 04 will show it.